In [1]:
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 

### Basic Qunatization Testing (8 bit quantization)

In [11]:
fp32_values = np.sort(np.random.uniform(low = 0.0, high = 5.0, size = (10)))
print(fp32_values)

beta = np.max(fp32_values)
alpha = np.min(fp32_values)
beta_q = pow(2, 8) - 1
alpha_q = 0 

print("BETA: {}\nALPHA:{}\nBETA_Q: {}\nALPHA_Q: {}".format(beta, alpha, beta_q, alpha_q))

[0.02592646 0.12784125 0.5927356  1.6261655  3.1386136  3.5052071
 3.76667453 3.77793591 4.06411028 4.42775709]
BETA: 4.427757090842791
ALPHA:0.02592646253587627
BETA_Q: 255
ALPHA_Q: 0


In [19]:
# scale 
s = (beta - alpha) / (beta_q - alpha_q) 
print("SCALE::", s)

# zero point 
z = round ((beta * alpha_q - alpha * beta_q) / (beta - alpha))
print("ZERO POINT::", z)

# clipping 
q = np.clip(z, alpha_q, beta_q)
print("CLIPPED VALUE", q)

# calculating the q values 
q_values = np.clip((fp32_values / s) + z, alpha_q, beta_q).astype(np.uint8)
print("FP32::", fp32_values)
print("Quantized::", q_values)

SCALE:: 6.716762994288418e-05
ZERO POINT:: -386
CLIPPED VALUE 0
FP32:: [0.02592646 0.12784125 0.5927356  1.6261655  3.1386136  3.5052071
 3.76667453 3.77793591 4.06411028 4.42775709]
Quantized:: [  0 237 246  16   6  87 140  52 216 254]


In [4]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fp32_text = [str(round(x,2)) for x in fp32_values.tolist()]
int8_text = [str(round(x,2)) for x in q_values.tolist()]    

fig = make_subplots(rows=1, cols=2)
fig.add_trace(go.Scatter(x=np.arange(len(fp32_values)), 
                         y=fp32_values, 
                         text=fp32_text,
                         mode='markers+text',
                         textposition='top left',
                         name='fp32_values'), row=1, col=1)
fig.add_trace(go.Scatter(x=np.arange(len(q_values)),
                         y=q_values,
                         text=int8_text,
                         mode='markers+text',
                         textposition='bottom right', 
                         name='Quantized values'), row=1, col=2)    


### Another way to quantize

In [41]:
import numpy as np 

x = (np.random.rand(20) * 5 - 2.5)

# 16 bit quantization
k_x = 15 - np.ceil(np.log(np.max(np.abs(x))))
x_q = np.round(x * 2 ** k_x)
x_f = x_q / 2 ** k_x

print("x:"); print(x); print('\n')
print("k_x:"); print(k_x); print('\n')
print("Quantized x:"); print(x_q); print('\n')
print("Reconstructed x:"); print(x_f); print('\n')
print("Quantization error:"); print(x - x_f); print('\n')

x:
[-0.27383869  0.61978229 -0.18017412  0.03730842  0.2210501  -0.77917557
 -2.24999673  1.507434    1.20550826  0.9059469   2.41418515  1.24485577
 -2.1750373   0.78552241 -0.04434591  1.95589953  1.97794851  0.05556455
  1.77319767  1.20697249]


k_x:
14.0


Quantized x:
[ -4487.  10155.  -2952.    611.   3622. -12766. -36864.  24698.  19751.
  14843.  39554.  20396. -35636.  12870.   -727.  32045.  32407.    910.
  29052.  19775.]


Reconstructed x:
[-0.27386475  0.61981201 -0.18017578  0.03729248  0.22106934 -0.7791748
 -2.25        1.50744629  1.20550537  0.90594482  2.41418457  1.24487305
 -2.17504883  0.78552246 -0.04437256  1.95587158  1.97796631  0.05554199
  1.77319336  1.20697021]


Quantization error:
[ 2.60592427e-05 -2.97220909e-05  1.65862536e-06  1.59392721e-05
 -1.92331827e-05 -7.69714376e-07  3.27336114e-06 -1.22938437e-05
  2.88907517e-06  2.07927140e-06  5.80797296e-07 -1.72808335e-05
  1.15282336e-05 -4.65648267e-08  2.66470522e-05  2.79518543e-05
 -1.77981514e-05

In [43]:
# 8 bit quantization 
k_x = 7 - np.ceil(np.log(np.max(np.abs(x))))
x_q = np.round(x * 2 ** k_x)
x_f = x_q / 2**k_x

print("x:"); print(x); print('\n')
print("k_x:"); print(k_x); print('\n')
print("Quantized x:"); print(x_q); print('\n')
print("Reconstructed x:"); print(x_f); print('\n')
print("Quantization error:"); print(x - x_f); print('\n')

x:
[-0.27383869  0.61978229 -0.18017412  0.03730842  0.2210501  -0.77917557
 -2.24999673  1.507434    1.20550826  0.9059469   2.41418515  1.24485577
 -2.1750373   0.78552241 -0.04434591  1.95589953  1.97794851  0.05556455
  1.77319767  1.20697249]


k_x:
6.0


Quantized x:
[ -18.   40.  -12.    2.   14.  -50. -144.   96.   77.   58.  155.   80.
 -139.   50.   -3.  125.  127.    4.  113.   77.]


Reconstructed x:
[-0.28125   0.625    -0.1875    0.03125   0.21875  -0.78125  -2.25
  1.5       1.203125  0.90625   2.421875  1.25     -2.171875  0.78125
 -0.046875  1.953125  1.984375  0.0625    1.765625  1.203125]


Quantization error:
[ 7.41131315e-03 -5.21771037e-03  7.32587738e-03  6.05841974e-03
  2.30010275e-03  2.07442560e-03  3.27336114e-06  7.43399522e-03
  2.38326017e-03 -3.03096510e-04 -7.68984889e-03 -5.14423396e-03
 -3.16229989e-03  4.27241437e-03  2.52908846e-03  2.77453389e-03
 -6.42648956e-03 -6.93544642e-03  7.57267353e-03  3.84749305e-03]


